# FGES — Runtime vs Graph Degree (fixed d=200)

固定 d=200、n=1000，扫描不同 ER degree，测量运行时间，拟合 runtime vs degree 的增长规律。

In [ ]:
import os, sys, time, warnings
import concurrent.futures
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "calm_dataset.py").exists() and (p / "coordinate_descent").exists():
            return p
    raise RuntimeError(f"repo root not found from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from calm_dataset import CalmDataset
print(f"REPO_ROOT: {REPO_ROOT}")

In [ ]:
ALGO_NAME = "FGES"
TAG       = "fges_degree"

import fges_compat as _tetrad_mod
_p = pd.DataFrame(np.eye(2), columns=["x0", "x1"])
_s = _tetrad_mod.TetradSearch(_p); _s.use_sem_bic(); _s.run_fges(); del _p, _s
print("FGES (Tetrad JVM): OK")

In [ ]:
D           = 200
DEGREE_LIST = [1, 2, 3, 4, 6, 8, 10, 15, 20]
N_SAMPLES   = 1000
TRIALS      = 3
GRAPH_TYPE  = "ER"
SEED_BASE   = 42
TIMEOUT_SEC = 3600
os.makedirs(str(REPO_ROOT / "experiments" / "results"), exist_ok=True)

In [ ]:
def run_with_timeout(fn, timeout):
    """Returns (elapsed_sec, error_str_or_None)."""
    ex = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    fut = ex.submit(fn)
    t0 = time.perf_counter()
    try:
        fut.result(timeout=timeout)
        elapsed = time.perf_counter() - t0
        ex.shutdown(wait=False)
        return elapsed, None
    except concurrent.futures.TimeoutError:
        ex.shutdown(wait=False)
        return float(timeout), "TIMEOUT"
    except Exception as e:
        elapsed = time.perf_counter() - t0
        ex.shutdown(wait=False)
        return elapsed, str(e)


def make_data(degree, seed):
    ds = CalmDataset(
        n=N_SAMPLES, d=D, graph_type=GRAPH_TYPE,
        degree=degree, sem_type="gauss",
        noise_ratio=4.0, seed=int(seed),
    )
    return ds.X

In [ ]:
def run_algo(X: np.ndarray):
    cols = [f"x{i}" for i in range(X.shape[1])]
    df_X = pd.DataFrame(X, columns=cols).astype("float64")
    search = _tetrad_mod.TetradSearch(df_X)
    search.set_verbose(False)
    search.use_sem_bic(penalty_discount=2)
    search.run_fges()

## 主循环

In [ ]:
records = []
stopped_at_deg = None
rng = np.random.default_rng(SEED_BASE)
seeds = rng.integers(0, 10**9, size=(len(DEGREE_LIST), TRIALS))

for gi, degree in enumerate(DEGREE_LIST):
    if stopped_at_deg is not None:
        break
    trial_times = []
    for t in range(TRIALS):
        X = make_data(degree, int(seeds[gi, t]))
        elapsed, err = run_with_timeout(lambda X=X: run_algo(X), TIMEOUT_SEC)
        if err == "TIMEOUT":
            print(f"degree={degree:5.1f}  trial={t+1}  TIMEOUT (>{TIMEOUT_SEC}s)")
            stopped_at_deg = degree
            break
        elif err:
            print(f"degree={degree:5.1f}  trial={t+1}  ERROR: {err}")
        else:
            trial_times.append(elapsed)
            print(f"degree={degree:5.1f}  trial={t+1}  {elapsed:.3f}s")
    if trial_times:
        records.append({"degree": degree, "mean_sec": np.mean(trial_times),
                        "std_sec": np.std(trial_times), "n_trials": len(trial_times)})

df = pd.DataFrame(records)
print()
print(df.to_string(index=False))

## 可视化

In [ ]:
if df.empty or len(df) < 3:
    print("Not enough data points to plot.")
else:
    deg_arr = df["degree"].values.astype(float)
    t_arr   = df["mean_sec"].values
    s_arr   = df["std_sec"].values

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"{ALGO_NAME} — runtime vs degree  (d={D}, n={N_SAMPLES}, ER)", fontsize=13)

    # log-log: power law
    ax = axes[0]
    ax.errorbar(deg_arr, t_arr, yerr=s_arr, fmt="o-", capsize=4, label="measured")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("degree  (log scale)"); ax.set_ylabel("time (s,  log scale)")
    ax.set_title("log-log  →  power law  t ∝ degᵏ ?")
    ax.grid(True, which="both", ls="--", alpha=0.4)
    sl, ic, r, *_ = stats.linregress(np.log(deg_arr), np.log(t_arr))
    xf = np.logspace(np.log10(deg_arr.min()), np.log10(deg_arr.max()), 100)
    ax.plot(xf, np.exp(ic) * xf**sl, "r--",
            label=f"fit: t ∝ deg^{sl:.2f}   R²={r**2:.3f}")
    ax.legend()

    # log-linear: exponential
    ax = axes[1]
    ax.errorbar(deg_arr, t_arr, yerr=s_arr, fmt="o-", capsize=4, label="measured")
    ax.set_yscale("log")
    ax.set_xlabel("degree"); ax.set_ylabel("time (s,  log scale)")
    ax.set_title("log-linear  →  exponential  t ∝ exp(k·deg) ?")
    ax.grid(True, which="both", ls="--", alpha=0.4)
    sl2, ic2, r2, *_ = stats.linregress(deg_arr, np.log(t_arr))
    xf2 = np.linspace(deg_arr.min(), deg_arr.max(), 100)
    ax.plot(xf2, np.exp(ic2 + sl2 * xf2), "r--",
            label=f"fit: t ∝ exp({sl2:.4f}·deg)   R²={r2**2:.3f}")
    ax.legend()

    plt.tight_layout()
    out_png = REPO_ROOT / "experiments" / "results" / f"{TAG}_scaling.png"
    plt.savefig(out_png, dpi=120)
    plt.show()
    print(f"figure saved → {out_png}")

## 结论

In [ ]:
if len(df) >= 3:
    deg_arr = df["degree"].values.astype(float)
    t_arr   = df["mean_sec"].values
    sl_ll, ic_ll, r_ll, *_ = stats.linregress(np.log(deg_arr), np.log(t_arr))
    sl_le, ic_le, r_le, *_ = stats.linregress(deg_arr, np.log(t_arr))

    print("=== Scaling law summary ===")
    print(f"  log-log    R² = {r_ll**2:.4f}  →  t ∝ deg^{sl_ll:.2f}  (power law / polynomial)")
    print(f"  log-linear R² = {r_le**2:.4f}  →  t ∝ exp({sl_le:.4f}·deg)  (exponential)")
    if r_ll**2 > r_le**2:
        print(f"  Better fit: POWER LAW  (exponent ≈ {sl_ll:.2f})")
    else:
        print(f"  Better fit: EXPONENTIAL  (rate ≈ {sl_le:.4f} per unit degree)")
    if stopped_at_deg is not None:
        print(f"  Practical max degree  <  {stopped_at_deg}  (timeout = {TIMEOUT_SEC}s, d={D}, n={N_SAMPLES})")
    else:
        print(f"  All degrees tested within timeout — practical max degree ≥ {DEGREE_LIST[-1]}  (d={D}, n={N_SAMPLES})")
else:
    print("Not enough data for scaling analysis.")